In [0]:
# Cell 1 — Setup (thay thế toàn bộ)
import subprocess
subprocess.run(["pip", "install", "boto3", "-q"])

import boto3
import pandas as pd
import io


AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""
BUCKET_NAME    = ""
REGION         = "eu-north-1"

# ── HÀM ĐÃ CẬP NHẬT: Ép buộc đọc duy nhất file snapshot gốc để đảm bảo số liệu chính xác ──
def read_csv_from_s3(table_name):
    """
    Ép buộc hệ thống bỏ qua dữ liệu incremental tăng trưởng tự động,
    chỉ lấy duy nhất file snapshot chuẩn (.csv) để trả về đúng kết quả ban đầu.
    """
    fallback_key = f"raw/{table_name}/{table_name}.csv"
    
    try:
        # Đọc trực tiếp file tổng thể ban đầu trên S3
        obj = s3.get_object(Bucket=BUCKET_NAME, Key=fallback_key)
        content = obj['Body'].read().decode('utf-8-sig')
        pandas_df = pd.read_csv(io.StringIO(content))
        spark_df = spark.createDataFrame(pandas_df)
        
        # In thông báo để theo dõi tiến trình gộp
        print(f"   [FIXED SNAPSHOT] Đã đọc file gốc chuẩn: {fallback_key} ({len(pandas_df)} rows)")
        return spark_df
        
    except Exception as e:
        # Trường hợp nếu bảng đó không tồn tại file .csv tổng ở ngoài, hệ thống sẽ báo lỗi
        raise FileNotFoundError(
            f"Không thể tìm thấy hoặc đọc file gốc tại s3://{BUCKET_NAME}/{fallback_key}\n"
            f"Chi tiết lỗi: {e}"
        )

print("    Kết nối S3 sẵn sàng")
print(f"   Mode: Fixed Snapshot Path (Đảm bảo số liệu khớp 100% kết quả ban đầu)")

    Kết nối S3 sẵn sàng
   Mode: Fixed Snapshot Path (Đảm bảo số liệu khớp 100% kết quả ban đầu)


In [0]:
# Test với bảng users trước
try:
    df_test = read_csv_from_s3("users")
    print(f"Đọc được! Số records: {df_test.count()}")
    df_test.show(3, truncate=False)
except Exception as e:
    print(f"Lỗi: {e}")

   [FIXED SNAPSHOT] Đã đọc file gốc chuẩn: raw/users/users.csv (461 rows)
Đọc được! Số records: 461
+---+--------+---------+-------------------+
|id |role    |is_active|date_joined        |
+---+--------+---------+-------------------+
|1  |ADMIN   |true     |2026-03-03 18:12:24|
|2  |PROVIDER|true     |2026-03-12 19:09:12|
|3  |PROVIDER|true     |2026-03-08 23:05:41|
+---+--------+---------+-------------------+
only showing top 3 rows


In [0]:
import pyspark.sql.functions as F

tables = ['users', 'tours', 'bookings', 'payments', 'revenues', 'reviews']

results = {}

for t in tables:
    print(f"\n{'='*50}")
    print(f"ĐANG ĐỌC: {t.upper()}")
    
    try:
        df = read_csv_from_s3(t)
        count = df.count()
        
        # Đếm null từng cột
        nulls = {}
        for c in df.columns:
            n = df.filter(F.col(c).isNull()).count()
            if n > 0:
                nulls[c] = n
        
        results[t] = {
            "status": "---OK---",
            "count": count,
            "columns": list(df.columns),
            "nulls": nulls
        }
        
        print(f"---OK--- {count} records")
        print(f"   Columns: {list(df.columns)}")
        df.printSchema()
        df.show(3, truncate=False)
        
        if nulls:
            print(f"  Null ở cột: {nulls}")
        else:
            print("   Không có null")
            
    except Exception as e:
        results[t] = {"status": "---LỖI---", "error": str(e)}
        print(f"---LỖI--- khi đọc {t}: {e}")

print(f"\n{'='*50}")
print("ĐỌC XONG 6 BẢNG")


ĐANG ĐỌC: USERS
   [FIXED SNAPSHOT] Đã đọc file gốc chuẩn: raw/users/users.csv (461 rows)
---OK--- 461 records
   Columns: ['id', 'role', 'is_active', 'date_joined']
root
 |-- id: long (nullable = true)
 |-- role: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- date_joined: string (nullable = true)

+---+--------+---------+-------------------+
|id |role    |is_active|date_joined        |
+---+--------+---------+-------------------+
|1  |ADMIN   |true     |2026-03-03 18:12:24|
|2  |PROVIDER|true     |2026-03-12 19:09:12|
|3  |PROVIDER|true     |2026-03-08 23:05:41|
+---+--------+---------+-------------------+
only showing top 3 rows
   Không có null

ĐANG ĐỌC: TOURS
   [FIXED SNAPSHOT] Đã đọc file gốc chuẩn: raw/tours/tours.csv (240 rows)
---OK--- 240 records
   Columns: ['id', 'creator_id', 'title', 'price', 'departure_date', 'slots', 'status', 'created_at', 'category_names']
root
 |-- id: long (nullable = true)
 |-- creator_id: long (nullable = true)
 |-- titl

In [0]:
# Schema mong đợi theo export_schema.md của [A]
EXPECTED_SCHEMAS = {
    "users":    ["id", "role", "is_active", "date_joined"],
    "tours":    ["id", "creator_id", "title", "price", "departure_date",
                 "slots", "status", "created_at", "category_names"],
    "bookings": ["id", "user_id", "tour_id", "number_of_people",
                 "total_price", "booking_date", "status", "created_at"],
    "payments": ["id", "booking_id", "amount", "payment_method",
                 "status", "created_at"],
    "revenues": ["id", "payment_id", "creator_id", "total_amount",
                 "creator_share", "admin_share", "created_at"],
    "reviews":  ["id", "user_id", "tour_id", "rating", "created_at"]
}

print("TỔNG KẾT VERIFY — NGÀY 1")
print("="*60)

schema_issues = []
encoding_issues = []
all_ok = True

for t, r in results.items():
    if r["status"] != "---OK---":
        print(f"  ---LỖI--- {t:12s} — KHÔNG ĐỌC ĐƯỢC: {r.get('error','')[:60]}")
        all_ok = False
        continue
    
    actual_cols   = [c.lower() for c in r["columns"]]
    expected_cols = EXPECTED_SCHEMAS.get(t, [])
    
    missing = [c for c in expected_cols if c not in actual_cols]
    extra   = [c for c in actual_cols   if c not in expected_cols]
    
    # Ghi nhận vấn đề schema
    if missing or extra:
        schema_issues.append({
            "table": t,
            "missing": missing,
            "extra": extra
        })
    
    warn = ""
    if missing: warn += f" |  ----THIẾU CỘT----: {missing}"
    if extra:   warn += f" |  -- cột thừa --: {extra}"
    if r["nulls"]: warn += f" | -- null --: {list(r['nulls'].keys())}"
    
    print(f" {t:12s} — {r['count']:5d} records{warn}")

print("="*60)

# Cảnh báo schema cho [D]
if schema_issues:
    print("\n  BÁO [D] (KHÁNH) — CÁC VẤN ĐỀ SCHEMA CẦN FIX:")
    for issue in schema_issues:
        t = issue["table"]
        if issue["missing"]:
            print(f"  ---LỖI--- Bảng [{t}] thiếu cột: {issue['missing']}")
            print(f"     -> Fix: bổ sung các cột này vào export_to_csv.py")
        if issue["extra"]:
            print(f"  ---LỖI---  Bảng [{t}] có cột thừa: {issue['extra']}")
            print(f"     -> Fix: bỏ các cột nhạy cảm này khỏi export_to_csv.py")
    all_ok = False

# Kiểm tra encoding tiếng Việt (bảng tours)
if "tours" in results and results["tours"]["status"] == "---OK---":
    print("\n Kiểm tra encoding tiếng Việt (bảng tours)...")
    df_tours = read_csv_from_s3("tours")
    
    # Kiểm tra cột title có tiếng Việt không bị lỗi
    if "title" in [c.lower() for c in df_tours.columns]:
        broken = df_tours.filter(
            F.col("title").contains("?") |
            F.col("title").contains("â€") |
            F.col("title").contains("Ã")
        ).count()
        
        if broken > 0:
            print(f"    {broken} dòng bị lỗi encoding tiếng Việt!")
            print(f"  -> Báo [D]: thêm encoding='utf-8-sig' vào export_to_csv.py")
            encoding_issues.append("tours.title")
        else:
            print("   Tiếng Việt hiển thị đúng")
    
    # Show thử title để kiểm tra bằng mắt
    print("\n  Kiểm tra bằng mắt — 3 tên tour:")
    df_tours.select("title").show(3, truncate=False)

# Kết luận cuối
print("\n" + "="*60)
if all_ok and not encoding_issues:
    print("KẾT LUẬN: Dữ liệu thật đã sẵn sàng!")
    print("Báo cả nhóm: có thể bắt đầu Ngày 2 (Bronze ingestion)")
else:
    print("KẾT LUẬN: Cần fix một số vấn đề trước khi sang Ngày 2")
    print("Gửi output này cho [D] (Khang) và [A] (Hà) để xử lý")
print("="*60)

TỔNG KẾT VERIFY — NGÀY 1
 users        —   461 records
 tours        —   240 records
 bookings     —   358 records
 payments     —   319 records
 revenues     —   114 records
 reviews      —   109 records |  -- cột thừa --: ['booking_id']

  BÁO [D] (KHÁNH) — CÁC VẤN ĐỀ SCHEMA CẦN FIX:
  ---LỖI---  Bảng [reviews] có cột thừa: ['booking_id']
     -> Fix: bỏ các cột nhạy cảm này khỏi export_to_csv.py

 Kiểm tra encoding tiếng Việt (bảng tours)...
   [FIXED SNAPSHOT] Đã đọc file gốc chuẩn: raw/tours/tours.csv (240 rows)
   Tiếng Việt hiển thị đúng

  Kiểm tra bằng mắt — 3 tên tour:
+------------------------------+
|title                         |
+------------------------------+
|Tour Hạ Long Cao Cấp 2N1Đ v26 |
|Hội An Phố Cổ Huyền Ảo v29    |
|Sapa Săn Mây Đỉnh Fansipan v12|
+------------------------------+
only showing top 3 rows

KẾT LUẬN: Cần fix một số vấn đề trước khi sang Ngày 2
Gửi output này cho [D] (Khang) và [A] (Hà) để xử lý


In [0]:
# Chạy cell này để kiểm tra bảng 'reviews' đang có những file nào trên S3
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="raw/reviews/")
if 'Contents' in response:
    for obj in response['Contents']:
        print(f"File đang có: {obj['Key']} - Size: {obj['Size']} bytes")

File đang có: raw/reviews/ - Size: 0 bytes
File đang có: raw/reviews/20260602_130601/data.csv - Size: 4055 bytes
File đang có: raw/reviews/20260602_130707/data.csv - Size: 4055 bytes
File đang có: raw/reviews/reviews.csv - Size: 4055 bytes
